# 01 · Area, 1 km grid, rain-fed mask and demo field

**Field Shift** (NASA Space Apps 2026), data notebook 1 of 6. *Code written with AI assistance (Claude, Anthropic); run and checked by the team.* Every later notebook uses the grid made here.

| Output file | What's in it |
|---|---|
| `data/grid.geojson` | About 2,800 squares of 1 km over the Bradano basin. Each one has its share of rain-fed arable land and of durum wheat (2018, 2022), plus its height and slope |
| `data/area.geojson` | The basin outline |
| `data/field.json` | The demo field: outline, grid square, land-cover checks. Later notebooks add their own sections to this file |

**How to run (about 3–5 minutes)**
1. Put your Earth Engine Cloud project ID into `EE_PROJECT` in the next cell.
2. **Runtime → Run all.** Allow Earth Engine and Google Drive when asked.
3. At the end, check the ✅ lines and look at the map. Then upload the three files that download to the repo's `data` folder. Copy the block under "Paste this back to Claude" into the chat.

**Optional, 2 minutes: draw the demo field.** By default the field is a 120 m square around the trial's coordinates. A hand-drawn outline makes the 30 m greenness data more exact.
1. Open https://geojson.io/#map=17/40.82460/16.09348 and switch the map to **Satellite** (bottom left). The trial's coordinates are at the centre of the screen. The close-up map at the end of this notebook shows them as a white dot.
2. Draw a polygon around the field that contains that point, using the polygon tool on the right.
3. Copy all the text in the right-hand panel, paste it between the triple quotes of `FIELD_OUTLINE` below, then run all again. It takes seconds the second time.

Only one field is drawn: it's the demo field shown in close detail. The grid and all later notebooks cover every farmland square in the basin.

In [ ]:
EE_PROJECT = "skilled-clover-514221-p6"    # Earth Engine Cloud project (change it if someone else runs this)

# Demo field: vetch trial of Denora et al. 2025, La Generale cooperative, Genzano di Lucania
FIELD_LAT, FIELD_LON = 40.82460, 16.09348
FIELD_OUTLINE = """"""                  # optional: paste the GeoJSON from geojson.io between the quotes
                                        # (left empty, an outline you drew in an earlier run is reused)

DRIVE_FOLDER = "field-shift"            # folder for this notebook's files (in Google Drive if USE_DRIVE is True)
USE_DRIVE = False                       # False: files stay in this Colab tab and download at the end. True: also saved in Google Drive

## Setup
Signs in to Earth Engine and connects Google Drive. Files are saved in Drive, so they survive if Colab disconnects.

In [ ]:
import sys, subprocess
try:
    import pyproj, shapely, folium          # normally already in Colab
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyproj", "shapely", "folium"], check=True)

import json, math, time, random, datetime as dt
import concurrent.futures as cf
from pathlib import Path

import ee
import pandas as pd
from pyproj import Transformer
from shapely.geometry import shape, box, Point, mapping
from shapely.ops import transform as shp_transform, unary_union
from shapely.prepared import prep

if EE_PROJECT == "your-cloud-project-id":
    raise ValueError("Put your Earth Engine Cloud project ID into EE_PROJECT (first code cell), then run again.")
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready")

IN_COLAB = "google.colab" in sys.modules
BASE = Path(DRIVE_FOLDER)
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:
        raise RuntimeError(
            f"Google Drive didn't connect ({str(e)[:80]}).\n"
            "Run this cell again. In the Google window, pick the same account and tick 'Select all' "
            "(the Google Drive box) before pressing Continue.\n"
            "If Drive can't be used at all, set USE_DRIVE = False in the first cell. The notebook will then ask you to "
            "upload notebook 01's files, and you upload its downloads to the repo yourself.") from None
    BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
elif IN_COLAB:
    BASE = Path("/content") / DRIVE_FOLDER
DATA, CACHE = BASE / "data", BASE / "cache"
for folder in (DATA, CACHE):
    folder.mkdir(parents=True, exist_ok=True)
print("Saving to", BASE)

TO_LAEA = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)    # lon/lat -> metres on the EU grid
TO_LONLAT = Transformer.from_crs("EPSG:3035", "EPSG:4326", always_xy=True)
CELL = 1000          # grid square size in metres
MIN_BASIN = 0.5      # keep squares at least half inside the basin
FARM_MIN = 0.10      # a square counts as farmland if 10% or more is rain-fed arable
NOTEBOOK = "notebooks/01_area_grid.ipynb"
RUN_AT = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%MZ")

def with_retries(fn, tries=6):
    """Earth Engine sometimes answers 'Too many requests' or times out: wait, then try again."""
    for k in range(tries):
        try:
            return fn()
        except Exception as e:
            if k == tries - 1:
                raise
            print("  retrying after:", str(e)[:100])
            time.sleep(min(60, 2 ** k + random.random()))

CHECKS = []
def check(ok, text, warn=False, fail=None):
    """Record and print a check: ✅ passed, ⚠️ worth a look, ❌ needs fixing (fail = what to print if it didn't pass)."""
    mark = "✅" if ok else ("⚠️" if warn else "❌")
    line = f"{mark} {text if ok or not fail else fail}"
    CHECKS.append(line)
    print(line)

def num(v, nd):
    """Round a value for the JSON files; empty or missing values become null."""
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    if math.isnan(v):
        return None
    return int(round(v)) if nd == 0 else round(v, nd)

def cell_id(x0, y0):
    """EEA reference grid code of the 1 km square whose lower-left corner is (x0, y0) in EPSG:3035."""
    return f"1kmE{int(x0) // 1000}N{int(y0) // 1000}"

SOURCES = {
    "hydrosheds": {"name": "HydroSHEDS HydroBASINS level 8", "ee_id": "WWF/HydroSHEDS/v1/Basins/hybas_8", "res_m": 500,
                   "licence": "HydroSHEDS licence: free for non-commercial and commercial use, with attribution",
                   "cite": "Lehner, B. & Grill, G. (2013). Hydrological Processes 27(15): 2171-2186. www.hydrosheds.org"},
    "worldcover": {"name": "ESA WorldCover 10 m 2021 v200", "ee_id": "ESA/WorldCover/v200", "res_m": 10, "licence": "CC BY 4.0",
                   "cite": "Zanaga, D. et al. (2022). ESA WorldCover 10 m 2021 v200. doi:10.5281/zenodo.7254221"},
    "corine": {"name": "CORINE Land Cover 2018", "ee_id": "COPERNICUS/CORINE/V20/100m/2018", "res_m": 100,
               "licence": "Copernicus Land Monitoring Service: free, full and open access, with attribution",
               "cite": "European Union, Copernicus Land Monitoring Service 2018, European Environment Agency (EEA)"},
    "eucropmap": {"name": "EU crop map 2018 and 2022 (JRC)", "ee_id": "JRC/D5/EUCROPMAP/V1", "res_m": 10, "licence": "CC BY 4.0",
                  "cite": "d'Andrimont, R. et al. (2021). Remote Sensing of Environment 266: 112708; Ghassemi, B. et al. (2024). Scientific Data 11: 1048"},
    "nasadem": {"name": "NASADEM elevation", "ee_id": "NASA/NASADEM_HGT/001", "res_m": 30, "licence": "NASA data: no restrictions on use",
                "cite": "NASA JPL (2020). NASADEM Merged DEM Global 1 arc second V001. doi:10.5067/MEaSUREs/NASADEM/NASADEM_HGT.001"},
}
print("Setup done")

## 1. The demo field outline
The hand-drawn outline if there is one, otherwise a 120 m square around the trial's coordinates. Everything for the field is worked out at the centre of this outline.

In [ ]:
def parse_outline(text):
    """Accepts what geojson.io gives (a FeatureCollection) or a bare Feature/Polygon; returns the first polygon."""
    text = (text or "").strip()
    if not text:
        return None
    g = json.loads(text)
    feats = g["features"] if g.get("type") == "FeatureCollection" else [g]
    for f in feats:
        geom = f.get("geometry", f)
        if geom and geom.get("type") in ("Polygon", "MultiPolygon"):
            s = shape(geom)
            if not s.is_valid:                     # e.g. a hand-drawn outline that crosses itself
                s = s.buffer(0)
            return max(s.geoms, key=lambda p: p.area) if s.geom_type == "MultiPolygon" else s
    raise ValueError("FIELD_OUTLINE has no polygon in it: draw a polygon (not a point or line) in geojson.io")

saved_field = DATA / "field.json"
if not FIELD_OUTLINE.strip() and saved_field.exists():           # reuse the outline drawn in an earlier run
    prev = json.loads(saved_field.read_text()).get("field", {})
    if prev.get("outline_source", "").startswith("drawn by hand"):
        FIELD_OUTLINE = json.dumps(prev["outline"])
        print("Using the outline you drew last time (saved in field.json). To change it, paste a new one into FIELD_OUTLINE.")

trial_pt = Point(*TO_LAEA.transform(FIELD_LON, FIELD_LAT))      # the trial's published coordinates, in metres
outline_ll = parse_outline(FIELD_OUTLINE)
if outline_ll is None:
    outline_m = box(trial_pt.x - 60, trial_pt.y - 60, trial_pt.x + 60, trial_pt.y + 60)
    outline_ll = shp_transform(TO_LONLAT.transform, outline_m)
    outline_source = "placeholder: 120 m square centred on the trial's coordinates (not yet drawn by hand)"
else:
    outline_m = shp_transform(TO_LAEA.transform, outline_ll)
    outline_source = "drawn by hand on satellite imagery"
field_pt = outline_m.centroid                                    # the field's data are worked out here
field_lon, field_lat = (round(v, 6) for v in TO_LONLAT.transform(field_pt.x, field_pt.y))
gap_m = outline_m.distance(trial_pt)
area_ha = outline_m.area / 1e4
if FIELD_OUTLINE.strip():
    check(gap_m <= 30, "The hand-drawn outline contains the trial's coordinates", warn=True,
          fail=f"The hand-drawn outline is {gap_m:.0f} m from the trial's coordinates: compare them on the close-up map at the end")
print(f"Field outline: {outline_source}; {area_ha:.1f} ha; centre {field_lat:.5f}, {field_lon:.5f}")

## 2. The Bradano basin
River basins from HydroSHEDS (built from NASA's SRTM radar elevation data). The notebook takes every sub-basin that drains to the same river mouth as a point on the main Bradano, where the Basentello joins it.

In [ ]:
REF_LON, REF_LAT = 16.3822, 40.6695          # Bradano-Basentello confluence, on the main river
HB = ee.FeatureCollection("WWF/HydroSHEDS/v1/Basins/hybas_8")

def get_basin():
    ref = ee.Geometry.Point([REF_LON, REF_LAT])
    main = HB.filterBounds(ref).first().get("MAIN_BAS")
    parts = HB.filter(ee.Filter.eq("MAIN_BAS", main))
    info = with_retries(lambda: ee.Dictionary({
        "main_bas": main, "parts": parts.size(), "area_km2": parts.aggregate_sum("SUB_AREA")}).getInfo())
    geom = with_retries(lambda: parts.geometry().dissolve(maxError=100).simplify(maxError=100).getInfo())
    return geom, info

basin_file = CACHE / "01_basin.json"
if basin_file.exists():
    saved = json.loads(basin_file.read_text())
    print("Using the saved basin outline")
else:
    geom, info = get_basin()
    saved = {"geom": geom, "info": info}
    basin_file.write_text(json.dumps(saved))
basin_info = saved["info"]

def polygonal(g):
    """Keep only the polygon parts of a shape and repair small self-overlaps."""
    if g.geom_type not in ("Polygon", "MultiPolygon"):
        g = unary_union([p for p in getattr(g, "geoms", []) if p.geom_type in ("Polygon", "MultiPolygon")])
    return g.buffer(0)

basin_ll = polygonal(shape(saved["geom"]))
basin = shp_transform(TO_LAEA.transform, basin_ll)          # metres, EPSG:3035
area_km2 = basin.area / 1e6
basin_source = f"HydroSHEDS HydroBASINS level 8, all sub-basins with MAIN_BAS = {basin_info['main_bas']}"
basin_ok = 2000 <= area_km2 <= 3800
check(basin_ok, f"Bradano basin: {area_km2:,.0f} km² from {basin_info['parts']} HydroSHEDS sub-basins "
                f"(published size 2,765 km²)")
if not basin_ok:
    print("   The basin looks wrong, so the notebook falls back to the planned box 15.8-16.9°E, 40.3-41.0°N.")
    basin_ll = box(15.8, 40.3, 16.9, 41.0)
    basin = shp_transform(TO_LAEA.transform, basin_ll)
    area_km2 = basin.area / 1e6
    basin_source = "Fallback box 15.8-16.9°E, 40.3-41.0°N (HydroSHEDS basin did not pass the size check)"

check(basin.contains(field_pt), "The demo field is inside the area",
      fail="The demo field is OUTSIDE the area: check FIELD_LAT, FIELD_LON and FIELD_OUTLINE")

## 3. The 1 km grid
Squares of the official EU reference grid (EEA, ETRS89-LAEA / EPSG:3035), so every square is exactly 1 km² and has a standard code such as `1kmE4836N1990`. A square is kept if at least half of it is inside the basin.

In [ ]:
minx, miny, maxx, maxy = basin.bounds
xs = range(int(math.floor(minx / CELL)) * CELL, int(math.ceil(maxx / CELL)) * CELL, CELL)
ys = range(int(math.floor(miny / CELL)) * CELL, int(math.ceil(maxy / CELL)) * CELL, CELL)
field_x0, field_y0 = int(field_pt.x // CELL) * CELL, int(field_pt.y // CELL) * CELL
FIELD_CELL = cell_id(field_x0, field_y0)

inside = prep(basin)
cells = []
for x0 in xs:
    for y0 in ys:
        square = box(x0, y0, x0 + CELL, y0 + CELL)
        if not inside.intersects(square):
            continue
        share = square.intersection(basin).area / CELL ** 2
        if share < MIN_BASIN and (x0, y0) != (field_x0, field_y0):
            continue
        corners = [(x0, y0), (x0 + CELL, y0), (x0 + CELL, y0 + CELL), (x0, y0 + CELL), (x0, y0)]
        ring = [[round(lon, 5), round(lat, 5)] for lon, lat in (TO_LONLAT.transform(x, y) for x, y in corners)]
        lon, lat = TO_LONLAT.transform(x0 + CELL / 2, y0 + CELL / 2)
        cells.append({"id": cell_id(x0, y0), "x0": x0, "y0": y0, "basin": round(share, 3),
                      "lon": round(lon, 5), "lat": round(lat, 5), "ring": ring})

check(2000 <= len(cells) <= 3800, f"{len(cells):,} grid squares of 1 km² (plan: about 2,800)")
print(f"The demo field is in square {FIELD_CELL}")

## 4. Land cover, durum wheat, height and slope for every square
For each square: the share that is cropland (ESA WorldCover 2021), non-irrigated arable land (CORINE 2018), both together (our **rain-fed arable** mask, which leaves out the irrigated orchards near the coast), any arable crop and durum wheat (EU crop map 2018 and 2022), plus mean height and slope (NASADEM). Takes 1–3 minutes; the result is saved, so a rerun takes seconds.

In [ ]:
WC = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
CLC = ee.Image("COPERNICUS/CORINE/V20/100m/2018").select("landcover")
ECM = ee.ImageCollection("JRC/D5/EUCROPMAP/V1").select("classification")
ECM18 = ECM.filterDate("2018-01-01", "2019-01-01").mosaic()
ECM22 = ECM.filterDate("2022-01-01", "2023-01-01").mosaic()
DEM = ee.Image("NASA/NASADEM_HGT/001").select("elevation")

n18, n22 = with_retries(lambda: ee.List([ECM.filterDate("2018-01-01", "2019-01-01").size(),
                                         ECM.filterDate("2022-01-01", "2023-01-01").size()]).getInfo())
check(n18 > 0 and n22 > 0, f"EU crop map found for 2018 ({n18} image) and 2022 ({n22} image)")

crop, arable = WC.eq(40), CLC.eq(211)
STACK = ee.Image.cat([
    crop.unmask(0).rename("crop"),
    arable.unmask(0).rename("arable"),
    crop.And(arable).unmask(0).rename("rainfed"),
    ECM22.gte(211).And(ECM22.lte(290)).unmask(0).rename("arable22"),
    ECM18.eq(212).unmask(0).rename("durum18"),
    ECM22.eq(212).unmask(0).rename("durum22"),
    DEM.rename("elev_m"),
    ee.Terrain.slope(DEM).rename("slope_deg"),
])
BANDS = ["crop", "arable", "rainfed", "arable22", "durum18", "durum22", "elev_m", "slope_deg"]

def square(c):
    return ee.Geometry.Rectangle([c["x0"], c["y0"], c["x0"] + CELL, c["y0"] + CELL], "EPSG:3035", False)

def reduce_batch(batch):
    fc = ee.FeatureCollection([ee.Feature(square(c), {"id": c["id"]}) for c in batch])
    out = STACK.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=20, crs="EPSG:3035", tileScale=4)
    feats = with_retries(lambda: out.select(["id"] + BANDS, None, False).getInfo())["features"]
    return [f["properties"] for f in feats]

stats_file = CACHE / "01_cells.csv"
if stats_file.exists() and len(pd.read_csv(stats_file)) == len(cells):
    stats = pd.read_csv(stats_file)
    print("Using saved land-cover numbers")
else:
    batches = [cells[i:i + 200] for i in range(0, len(cells), 200)]
    rows, t0 = [], time.time()
    with cf.ThreadPoolExecutor(max_workers=6) as pool:
        for i, part in enumerate(pool.map(reduce_batch, batches), 1):
            rows += part
            print(f"  batch {i}/{len(batches)} done, {time.time() - t0:.0f} s")
    stats = pd.DataFrame(rows)
    stats.to_csv(stats_file, index=False)

stats = stats.set_index("id").reindex([c["id"] for c in cells])
check(stats["rainfed"].notna().all(), f"Land-cover numbers for all {len(stats):,} squares",
      fail=f"{stats['rainfed'].isna().sum()} squares have no land-cover numbers: run all again")
w = pd.Series([c["basin"] for c in cells], index=stats.index)
rainfed_share = (stats["rainfed"] * w).sum() / w.sum()
durum_share = (stats["durum22"] * w).sum() / w.sum()
farm_squares = int((stats["rainfed"].round(2) >= FARM_MIN).sum())     # same rule as the farm flag (share rounded to 2 decimals)
check(0.25 <= rainfed_share <= 0.85, f"Rain-fed arable land covers {rainfed_share:.0%} of the area", warn=True)
check(0.05 <= durum_share <= 0.6, f"Durum wheat covered {durum_share:.0%} of the area in 2022 (EU crop map)", warn=True)
print(f"{farm_squares:,} squares are at least {FARM_MIN:.0%} rain-fed arable (used as farmland in later notebooks)")

## 5. The demo field
Checks that the field really is rain-fed arable land before any data is worked out for it (lesson from the NSW practice run, where the default point was on grassland).

In [ ]:
WC_NAMES = {10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland", 50: "Built-up", 60: "Bare or sparse",
            70: "Snow and ice", 80: "Water", 90: "Wetland", 95: "Mangroves", 100: "Moss and lichen"}
CLC_NAMES = {111: "Continuous urban", 112: "Discontinuous urban", 121: "Industrial or commercial", 211: "Non-irrigated arable land",
             212: "Permanently irrigated land", 221: "Vineyards", 222: "Fruit trees", 223: "Olive groves", 231: "Pastures",
             241: "Annual and permanent crops", 242: "Complex cultivation patterns", 243: "Farmland with natural vegetation",
             311: "Broad-leaved forest", 321: "Natural grassland", 323: "Sclerophyllous vegetation",
             324: "Woodland-shrub", 333: "Sparsely vegetated", 512: "Water bodies"}
ECM_NAMES = {100: "Artificial", 211: "Common wheat", 212: "Durum wheat", 213: "Barley", 214: "Rye", 215: "Oats", 216: "Maize",
             217: "Rice", 218: "Triticale", 219: "Other cereals", 221: "Potatoes", 222: "Sugar beet", 223: "Other root crops",
             230: "Other industrial crops", 231: "Sunflower", 232: "Rapeseed", 233: "Soya", 240: "Dry pulses",
             250: "Fodder crops", 290: "Bare arable land", 300: "Woodland and shrubland", 500: "Grassland",
             600: "Bare land", 700: "Water", 800: "Wetland"}

fgeom = ee.Geometry.Polygon([[[x, y] for x, y in outline_m.exterior.coords]], "EPSG:3035", False)
CLASSES = ee.Image.cat([WC.rename("wc"), CLC.rename("clc"), ECM18.rename("ecm18"), ECM22.rename("ecm22")])
field_res = with_retries(lambda: ee.Dictionary({
    "mode": CLASSES.reduceRegion(ee.Reducer.mode(), fgeom, scale=10, crs="EPSG:3035", maxPixels=1e9),
    "mean": STACK.reduceRegion(ee.Reducer.mean(), fgeom, scale=10, crs="EPSG:3035", maxPixels=1e9),
}).getInfo())
mode, mean = field_res["mode"], field_res["mean"]

def cls(v):
    return None if v is None else int(round(v))

landcover = {
    "worldcover_2021": {"class": cls(mode.get("wc")), "name": WC_NAMES.get(cls(mode.get("wc")), "?"), "share_cropland": num(mean.get("crop"), 2)},
    "corine_2018": {"class": cls(mode.get("clc")), "name": CLC_NAMES.get(cls(mode.get("clc")), "other"), "share_non_irrigated_arable": num(mean.get("arable"), 2)},
    "eu_cropmap_2018": {"class": cls(mode.get("ecm18")), "name": ECM_NAMES.get(cls(mode.get("ecm18")), "?"), "share_durum": num(mean.get("durum18"), 2)},
    "eu_cropmap_2022": {"class": cls(mode.get("ecm22")), "name": ECM_NAMES.get(cls(mode.get("ecm22")), "?"), "share_durum": num(mean.get("durum22"), 2)},
}
is_crop = landcover["worldcover_2021"]["class"] == 40
is_arable = landcover["corine_2018"]["class"] == 211
check(is_crop, f"Demo field is {landcover['worldcover_2021']['name'].lower()} (ESA WorldCover 2021)")
check(is_arable, f"Demo field is '{landcover['corine_2018']['name']}' (CORINE 2018)", warn=True)
for year in ("2018", "2022"):
    c = landcover[f"eu_cropmap_{year}"]
    check(c["class"] is not None and 211 <= c["class"] <= 290, f"EU crop map {year} at the field: {c['name']}", warn=True)
print(f"Demo field: height {num(mean.get('elev_m'), 0)} m; slope {num(mean.get('slope_deg'), 1)}°")

## 6. Write the data files
Writes `grid.geojson`, `area.geojson` and `field.json` to the Drive folder. If `field.json` already has sections from later notebooks, they're kept.

In [ ]:
FIELDS = {
    "id": "EEA 1 km reference grid code: lower-left corner in km, ETRS89-LAEA (EPSG:3035)",
    "lon, lat": "centre of the square (WGS84)",
    "basin": "share of the square inside the area (0-1)",
    "crop": "share that is cropland: ESA WorldCover 2021, class 40",
    "arable": "share that is non-irrigated arable land: CORINE 2018, class 211",
    "rainfed": "share that is both: the project's rain-fed arable mask",
    "arable22": "share in any arable crop: EU crop map 2022, classes 211-290",
    "durum18, durum22": "share mapped as durum wheat: EU crop map 2018 and 2022, class 212",
    "elev_m": "mean height in metres (NASADEM, 30 m)",
    "slope_deg": "mean slope in degrees (NASADEM, 30 m)",
    "farm": f"true if rain-fed arable covers at least {FARM_MIN:.0%} of the square; later notebooks work these out first",
}
ROUND = {"crop": 2, "arable": 2, "rainfed": 2, "arable22": 2, "durum18": 2, "durum22": 2, "elev_m": 0, "slope_deg": 1}

features = []
for c in cells:
    s = stats.loc[c["id"]]
    props = {"id": c["id"], "lon": c["lon"], "lat": c["lat"], "basin": round(c["basin"], 2)}
    props.update({b: num(s[b], ROUND[b]) for b in BANDS})
    props["farm"] = bool((props["rainfed"] or 0) >= FARM_MIN)
    features.append({"type": "Feature", "id": c["id"], "geometry": {"type": "Polygon", "coordinates": [c["ring"]]},
                     "properties": props})

grid_meta = {"project": "Field Shift (NASA Space Apps Challenge 2026)", "made_by": NOTEBOOK, "made_at": RUN_AT,
             "area": basin_source, "grid": "EEA 1 km reference grid, ETRS89-LAEA (EPSG:3035)",
             "squares": len(features), "field_square": FIELD_CELL, "fields": FIELDS,
             "sources": {k: SOURCES[k] for k in ("hydrosheds", "worldcover", "corine", "eucropmap", "nasadem")}}
grid = {"type": "FeatureCollection", "meta": grid_meta, "features": features}

def round_coords(geom):
    """Basin outline coordinates to 5 decimals (about 1 m)."""
    g = mapping(geom)
    def r(x):
        return [r(i) for i in x] if isinstance(x[0], (list, tuple)) else [round(x[0], 5), round(x[1], 5)]
    return {"type": g["type"], "coordinates": r(g["coordinates"])}

area = {"type": "FeatureCollection",
        "meta": {"made_by": NOTEBOOK, "made_at": RUN_AT, "sources": {"hydrosheds": SOURCES["hydrosheds"]}},
        "features": [{"type": "Feature", "id": "bradano",
                      "properties": {"name": "Bradano river basin", "area_km2": round(area_km2), "source": basin_source},
                      "geometry": round_coords(basin_ll)}]}

field_file = DATA / "field.json"
field_doc = json.loads(field_file.read_text()) if field_file.exists() else {}
field_doc.setdefault("meta", {}).update({"project": "Field Shift (NASA Space Apps Challenge 2026)", "updated": RUN_AT})
field_doc["meta"].setdefault("made_by", {})["field"] = f"{NOTEBOOK} ({RUN_AT})"
field_doc["meta"].setdefault("sources", {}).update({k: SOURCES[k] for k in ("worldcover", "corine", "eucropmap", "nasadem")})
field_doc["field"] = {
    "name": "Demo field, Genzano di Lucania",
    "about": "Vetch and durum trial field of Denora et al. (2025), La Generale cooperative",
    "lon": field_lon, "lat": field_lat,
    "trial_lon": FIELD_LON, "trial_lat": FIELD_LAT, "trial_to_outline_m": round(gap_m),
    "outline": round_coords(outline_ll), "outline_source": outline_source, "area_ha": round(area_ha, 2),
    "square": FIELD_CELL, "elev_m": num(mean.get("elev_m"), 0), "slope_deg": num(mean.get("slope_deg"), 1),
    "landcover": landcover,
    "checks": {"inside_area": bool(basin.contains(field_pt)), "cropland": bool(is_crop), "non_irrigated_arable": bool(is_arable)},
}

def write_json(path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, separators=(",", ":")))
    return path.stat().st_size / 1e6

for name, obj in (("grid.geojson", grid), ("area.geojson", area), ("field.json", field_doc)):
    mb = write_json(DATA / name, obj)
    check(mb < 5, f"{name} written: {mb:.2f} MB")

## 7. Look at it
Two satellite maps. The first shows the whole basin, with squares shaded by their share of rain-fed arable land (darker means more). The second is a close-up of the demo field: the red line is its outline and the white dot is the trial's published coordinates.

In [ ]:
try:
    import folium
    SAT = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
    LABELS = "https://server.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}"
    def base_map(zoom):
        m = folium.Map(location=[field_lat, field_lon], zoom_start=zoom, tiles=None)
        folium.TileLayer(SAT, attr="Esri, Maxar, Earthstar Geographics", name="Satellite", max_zoom=19).add_to(m)
        folium.TileLayer(LABELS, attr="Esri", name="Place names", overlay=True, max_zoom=19).add_to(m)
        return m
    def add_field(m):
        folium.GeoJson(field_doc["field"]["outline"], name="Demo field",
                       style_function=lambda f: {"color": "#ff0000", "weight": 3, "fillOpacity": 0}).add_to(m)
        folium.CircleMarker([FIELD_LAT, FIELD_LON], radius=6, color="#ff0000", weight=2, fill=True, fill_color="#ffffff",
                            fill_opacity=1, tooltip="Trial coordinates (Denora et al. 2025)").add_to(m)

    m1 = base_map(10)
    def shade(f):
        r = f["properties"]["rainfed"] or 0
        return {"fillColor": "#e3b505", "color": "#e3b505", "weight": 0.3, "fillOpacity": 0.1 + 0.6 * r}
    folium.GeoJson({"type": "FeatureCollection", "features": features}, name="Grid: rain-fed arable share",
                   style_function=shade, tooltip=folium.GeoJsonTooltip(["id", "rainfed", "durum22", "elev_m"])).add_to(m1)
    folium.GeoJson(area["features"][0], name="Basin",
                   style_function=lambda f: {"color": "#3b82f6", "weight": 2, "fillOpacity": 0}).add_to(m1)
    add_field(m1)
    folium.LayerControl().add_to(m1)
    bw, bs, be, bn = basin_ll.bounds
    m1.fit_bounds([[bs, bw], [bn, be]])
    display(m1)

    m2 = base_map(16)
    add_field(m2)
    display(m2)
    print(f"Close-up: the trial's coordinates are {gap_m:.0f} m from the outline" if gap_m > 0 else
          "Close-up: the trial's coordinates are inside the outline")
except Exception as e:
    print("Maps not shown (the data files are fine):", str(e)[:150])

## 8. Checks, downloads and what to paste back

In [ ]:
print("\n".join(CHECKS))
bad = [c for c in CHECKS if c.startswith("❌")]
print("\nAll essential checks passed." if not bad else f"\n{len(bad)} check(s) failed: paste the lines above into the chat.")

if IN_COLAB:
    from google.colab import files
    for name in ("grid.geojson", "area.geojson", "field.json"):
        files.download(str(DATA / name))
    print("Downloading grid.geojson, area.geojson and field.json: upload all three to the repo's data folder.")

In [ ]:
print("=== Paste this back to Claude ===")
print(f"01_area_grid run {RUN_AT}")
print(f"area: {basin_source}; {area_km2:,.0f} km2")
print(f"squares: {len(features)}; farm squares: {farm_squares}; field square: {FIELD_CELL}")
print(f"rain-fed arable {rainfed_share:.1%}; durum 2022 {durum_share:.1%}; durum 2018 {(stats['durum18'] * w).sum() / w.sum():.1%}")
print("field:", json.dumps({k: field_doc["field"][k] for k in ("lat", "lon", "trial_to_outline_m", "area_ha", "elev_m", "slope_deg", "landcover", "checks", "outline_source")}))
print("\n".join(CHECKS))